# March INN Perimeter Validation

Отдельная проверка за март 2026:
- считаем ИНН, которые должны быть в отчете по Lake-логике,
- сравниваем с Excel,
- выводим лишние ИНН относительно Excel (`expected - excel`).

In [ ]:
# Импорты и настройки отображения
import re
import pandas as pd
from rail_connectors.connection import connect

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)

In [ ]:
# Параметры запуска (март)
month_start = '2026-03-01'
month_end = '2026-03-31'

excel_path = '/home/jovyan/documents/Equaring/Data/03_Март_2026.xlsx'
excel_header = 0
inn_aliases = ['ИНН', 'inn', 'c_inn', 'INN']

In [ ]:
# Подключение к Impala
imp = connect(
    to='IMPALA',
    extra_options={'db': 'sandbox_ai'},
    driver_args={'tez.queue.name': 'ai'},
    kerberos={
        'keytab_path': '/home/jovyan/test_requests/tech.keytab',
        'use_credentials': True,
        'update_keytab': True,
    },
    user_params={'user_name': 'Shestopalov-VYur'}
)
imp._init_connection()
print('Impala connection initialized')

In [ ]:
# Вспомогательные функции: выбор колонки и нормализация ИНН
def pick_col(columns, candidates):
    cols = list(columns)
    lower_map = {str(c).strip().lower(): c for c in cols}
    for cand in candidates:
        if cand in cols:
            return cand
        key = str(cand).strip().lower()
        if key in lower_map:
            return lower_map[key]
    return None


def normalize_inn(value):
    if pd.isna(value):
        return None

    s = str(value).strip()
    s = re.sub(r'\.0$', '', s)
    s = re.sub(r'\D+', '', s)
    if not s:
        return None

    # Восстановление выпавшего ведущего 0
    if len(s) == 9:
        s = s.zfill(10)
    elif len(s) == 11:
        s = s.zfill(12)

    if len(s) not in (10, 12):
        return None
    return s

In [ ]:
# Шаг 1: загружаем Excel марта и строим множество ИНН
excel_raw = pd.read_excel(excel_path, header=excel_header)
excel_inn_col = pick_col(excel_raw.columns, inn_aliases)

if excel_inn_col is None:
    raise ValueError(f'В Excel не найдена колонка ИНН. Доступные: {list(excel_raw.columns)}')

excel_df = pd.DataFrame({'inn_raw': excel_raw[excel_inn_col]})
excel_df['inn_norm'] = excel_df['inn_raw'].apply(normalize_inn)
excel_inn_set = set(excel_df['inn_norm'].dropna().drop_duplicates().tolist())

print(f'Excel INN column: {excel_inn_col}')
print(f'Уникальных ИНН в Excel марта: {len(excel_inn_set)}')

In [ ]:
# Шаг 2: строим expected INN по Lake-логике (agreements + companies + agr_terms)
with imp:
    expected_lake = imp.fetch(f"""
        select
            cast(a.n_agr as string) as n_agr,
            cast(a.abs_agr_id as string) as agr_id,
            cast(a.d_valid_from as date) as agr_d_valid_from,
            cast(a.d_valid_to as date) as agr_d_valid_to,
            cast(c.c_inn as string) as c_inn
        from ods_alpha.scd1_agreements a
        join ods_alpha.scd1_companies c
          on c.n_cmp = a.n_cmp_client
        where upper(trim(cast(a.acq_class as string))) = 'SA'
          and cast(a.d_valid_from as date) <= cast('{month_end}' as date)
          and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('{month_start}' as date))
          and coalesce(a.ods_deleted_flg, '0') <> '1'
          and coalesce(c.ods_deleted_flg, '0') <> '1'
          and c.c_inn is not null
          and exists (
                select 1
                from ods_alpha.scd1_agr_terms t
                where cast(t.n_agr as string) = cast(a.n_agr as string)
                  and cast(t.d_valid_from as date) <= cast('{month_end}' as date)
                  and (t.d_valid_to is null or cast(t.d_valid_to as date) > cast('{month_start}' as date))
                  and upper(trim(cast(t.cf_ter_type as string))) = 'P'
                  and upper(trim(cast(t.c_fiid_grp as string))) = 'RSHB'
                  and coalesce(t.ods_deleted_flg, '0') <> '1'
          )
    """)

if expected_lake is None:
    expected_lake = pd.DataFrame(columns=['n_agr', 'agr_id', 'agr_d_valid_from', 'agr_d_valid_to', 'c_inn'])

expected_lake = expected_lake.copy()
expected_lake['inn_norm'] = expected_lake['c_inn'].apply(normalize_inn)
expected_lake = expected_lake[expected_lake['inn_norm'].notna()].copy()
expected_inn_set = set(expected_lake['inn_norm'].dropna().drop_duplicates().tolist())

print(f'Уникальных expected ИНН по Lake-правилам: {len(expected_inn_set)}')

In [ ]:
# Шаг 3: сравниваем expected vs Excel и считаем KPI
intersect_inn = expected_inn_set & excel_inn_set
extra_vs_excel = expected_inn_set - excel_inn_set
missing_vs_expected = excel_inn_set - expected_inn_set

kpi_df = pd.DataFrame([
    {'metric': 'expected_inn_cnt', 'value': len(expected_inn_set)},
    {'metric': 'excel_inn_cnt', 'value': len(excel_inn_set)},
    {'metric': 'intersect_inn_cnt', 'value': len(intersect_inn)},
    {'metric': 'extra_vs_excel_cnt', 'value': len(extra_vs_excel)},
    {'metric': 'missing_vs_expected_cnt', 'value': len(missing_vs_expected)},
])

display(kpi_df)

In [ ]:
# Шаг 4: выводим лишние ИНН относительно Excel + детали по ним
extra_vs_excel_df = pd.DataFrame({'inn_norm': sorted(list(extra_vs_excel))})
print('Лишние ИНН относительно Excel (expected - excel):')
display(extra_vs_excel_df)

extra_details_df = (
    expected_lake[
        expected_lake['inn_norm'].isin(extra_vs_excel)
    ][['inn_norm', 'n_agr', 'agr_id', 'agr_d_valid_from', 'agr_d_valid_to']]
    .drop_duplicates()
    .sort_values(['inn_norm', 'n_agr'])
)

print('Детали по лишним ИНН (top 20 строк):')
display(extra_details_df.head(20))

In [ ]:
# Шаг 5: funnel-проверка — где именно фильтры сильнее всего срезают ИНН
with imp:
    funnel_df = imp.fetch(f"""
        with base_agreements as (
            select
                cast(a.n_agr as string) as n_agr,
                regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', '') as inn_raw
            from ods_alpha.scd1_agreements a
            join ods_alpha.scd1_companies c
              on c.n_cmp = a.n_cmp_client
            where upper(trim(cast(a.acq_class as string))) = 'SA'
              and cast(a.d_valid_from as date) <= cast('{month_end}' as date)
              and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('{month_start}' as date))
              and coalesce(a.ods_deleted_flg, '0') <> '1'
              and coalesce(c.ods_deleted_flg, '0') <> '1'
              and c.c_inn is not null
        ),
        terms_active as (
            select distinct cast(t.n_agr as string) as n_agr
            from ods_alpha.scd1_agr_terms t
            where cast(t.d_valid_from as date) <= cast('{month_end}' as date)
              and (t.d_valid_to is null or cast(t.d_valid_to as date) > cast('{month_start}' as date))
              and coalesce(t.ods_deleted_flg, '0') <> '1'
        ),
        terms_active_p as (
            select distinct cast(t.n_agr as string) as n_agr
            from ods_alpha.scd1_agr_terms t
            where cast(t.d_valid_from as date) <= cast('{month_end}' as date)
              and (t.d_valid_to is null or cast(t.d_valid_to as date) > cast('{month_start}' as date))
              and upper(trim(cast(t.cf_ter_type as string))) = 'P'
              and coalesce(t.ods_deleted_flg, '0') <> '1'
        ),
        terms_active_p_rshb as (
            select distinct cast(t.n_agr as string) as n_agr
            from ods_alpha.scd1_agr_terms t
            where cast(t.d_valid_from as date) <= cast('{month_end}' as date)
              and (t.d_valid_to is null or cast(t.d_valid_to as date) > cast('{month_start}' as date))
              and upper(trim(cast(t.cf_ter_type as string))) = 'P'
              and upper(trim(cast(t.c_fiid_grp as string))) = 'RSHB'
              and coalesce(t.ods_deleted_flg, '0') <> '1'
        ),
        s1 as (
            select distinct b.inn_raw
            from base_agreements b
        ),
        s2 as (
            select distinct b.inn_raw
            from base_agreements b
            join terms_active t on t.n_agr = b.n_agr
        ),
        s3 as (
            select distinct b.inn_raw
            from base_agreements b
            join terms_active_p t on t.n_agr = b.n_agr
        ),
        s4 as (
            select distinct b.inn_raw
            from base_agreements b
            join terms_active_p_rshb t on t.n_agr = b.n_agr
        )
        select '01_base_agreements_sa_active' as stage, count(*) as inn_cnt from s1
        union all
        select '02_plus_terms_active' as stage, count(*) as inn_cnt from s2
        union all
        select '03_plus_cf_ter_type_P' as stage, count(*) as inn_cnt from s3
        union all
        select '04_plus_c_fiid_grp_RSHB' as stage, count(*) as inn_cnt from s4
    """)

if funnel_df is None:
    funnel_df = pd.DataFrame(columns=['stage', 'inn_cnt'])

display(funnel_df)

In [ ]:
# Шаг 6: пересечение с Excel за март БЕЗ фильтра c_fiid_grp='RSHB' (оставляем active + cf_ter_type='P')
with imp:
    expected_lake_no_rshb = imp.fetch(f"""
        select
            cast(a.n_agr as string) as n_agr,
            cast(a.abs_agr_id as string) as agr_id,
            cast(a.d_valid_from as date) as agr_d_valid_from,
            cast(a.d_valid_to as date) as agr_d_valid_to,
            cast(c.c_inn as string) as c_inn
        from ods_alpha.scd1_agreements a
        join ods_alpha.scd1_companies c
          on c.n_cmp = a.n_cmp_client
        where upper(trim(cast(a.acq_class as string))) = 'SA'
          and cast(a.d_valid_from as date) <= cast('{month_end}' as date)
          and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('{month_start}' as date))
          and coalesce(a.ods_deleted_flg, '0') <> '1'
          and coalesce(c.ods_deleted_flg, '0') <> '1'
          and c.c_inn is not null
          and exists (
                select 1
                from ods_alpha.scd1_agr_terms t
                where cast(t.n_agr as string) = cast(a.n_agr as string)
                  and cast(t.d_valid_from as date) <= cast('{month_end}' as date)
                  and (t.d_valid_to is null or cast(t.d_valid_to as date) > cast('{month_start}' as date))
                  and upper(trim(cast(t.cf_ter_type as string))) = 'P'
                  and coalesce(t.ods_deleted_flg, '0') <> '1'
          )
    """)

if expected_lake_no_rshb is None:
    expected_lake_no_rshb = pd.DataFrame(columns=['n_agr', 'agr_id', 'agr_d_valid_from', 'agr_d_valid_to', 'c_inn'])

expected_lake_no_rshb = expected_lake_no_rshb.copy()
expected_lake_no_rshb['inn_norm'] = expected_lake_no_rshb['c_inn'].apply(normalize_inn)
expected_lake_no_rshb = expected_lake_no_rshb[expected_lake_no_rshb['inn_norm'].notna()].copy()
lake_inn_no_rshb_set = set(expected_lake_no_rshb['inn_norm'].dropna().drop_duplicates().tolist())

intersect_no_rshb = lake_inn_no_rshb_set & excel_inn_set
only_lake_no_rshb = lake_inn_no_rshb_set - excel_inn_set
only_excel_no_rshb = excel_inn_set - lake_inn_no_rshb_set

kpi_no_rshb_df = pd.DataFrame([
    {'metric': 'lake_inn_no_rshb_cnt', 'value': len(lake_inn_no_rshb_set)},
    {'metric': 'excel_inn_cnt', 'value': len(excel_inn_set)},
    {'metric': 'intersect_inn_no_rshb_cnt', 'value': len(intersect_no_rshb)},
    {'metric': 'only_lake_no_rshb_cnt', 'value': len(only_lake_no_rshb)},
    {'metric': 'only_excel_no_rshb_cnt', 'value': len(only_excel_no_rshb)},
])

display(kpi_no_rshb_df)

extra_no_rshb_df = pd.DataFrame({'inn_norm': sorted(list(only_lake_no_rshb))})
print('ИНН только в Lake (без RSHB-фильтра):')
display(extra_no_rshb_df.head(50))

In [ ]:
# Шаг 7: параметры Q1 и нормализация agr_id для проверки пересечений по ИНН и agr_id.
from decimal import Decimal, InvalidOperation

q1_sources = [
    {'report_month': '2026-01-01', 'excel_path': '/home/jovyan/documents/Equaring/Data/01_Январь_2026.xlsx', 'header': 1},
    {'report_month': '2026-02-01', 'excel_path': '/home/jovyan/documents/Equaring/Data/02_Февраль_2026.xlsx', 'header': 0},
    {'report_month': '2026-03-01', 'excel_path': '/home/jovyan/documents/Equaring/Data/03_Март_2026.xlsx', 'header': 0},
]

agr_aliases = ['ID договора', 'agr_id', 'abs_agr_id']


def normalize_agr(value):
    if pd.isna(value):
        return None

    s = str(value).strip().replace('\xa0', '').replace(' ', '').replace(',', '.')
    if s in {'', 'nan', 'None'}:
        return None

    try:
        d = Decimal(s)
        if d == d.to_integral_value():
            return str(int(d))
    except (InvalidOperation, ValueError):
        pass

    s = re.sub(r'\.0$', '', s)
    if s in {'', 'nan', 'None'}:
        return None
    return s

In [ ]:
# Шаг 8: пересечения по ИНН и agr_id для Jan/Feb/Mar (без фильтра RSHB, но с active + cf_ter_type='P').
q1_kpi_rows = []
q1_only_lake_inn = {}
q1_only_excel_inn = {}
q1_only_lake_agr = {}
q1_only_excel_agr = {}

for src in q1_sources:
    report_month = src['report_month']
    report_ts = pd.to_datetime(report_month)
    m_start = report_ts.strftime('%Y-%m-%d')
    m_end = (report_ts + pd.offsets.MonthEnd(1)).strftime('%Y-%m-%d')

    # Excel sets
    excel_raw_m = pd.read_excel(src['excel_path'], header=src['header'])
    excel_inn_col_m = pick_col(excel_raw_m.columns, inn_aliases)
    excel_agr_col_m = pick_col(excel_raw_m.columns, agr_aliases)

    if excel_inn_col_m is None:
        raise ValueError(f"[{report_month}] В Excel не найдена колонка ИНН. Доступные: {list(excel_raw_m.columns)}")
    if excel_agr_col_m is None:
        raise ValueError(f"[{report_month}] В Excel не найдена колонка agr_id. Доступные: {list(excel_raw_m.columns)}")

    excel_inn_set_m = set(excel_raw_m[excel_inn_col_m].apply(normalize_inn).dropna().drop_duplicates().tolist())
    excel_agr_set_m = set(excel_raw_m[excel_agr_col_m].apply(normalize_agr).dropna().drop_duplicates().tolist())

    # Lake sets (no RSHB)
    with imp:
        lake_m = imp.fetch(f"""
            select
                cast(a.abs_agr_id as string) as agr_id,
                cast(c.c_inn as string) as c_inn
            from ods_alpha.scd1_agreements a
            join ods_alpha.scd1_companies c
              on c.n_cmp = a.n_cmp_client
            where upper(trim(cast(a.acq_class as string))) = 'SA'
              and cast(a.d_valid_from as date) <= cast('{m_end}' as date)
              and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('{m_start}' as date))
              and coalesce(a.ods_deleted_flg, '0') <> '1'
              and coalesce(c.ods_deleted_flg, '0') <> '1'
              and c.c_inn is not null
              and exists (
                    select 1
                    from ods_alpha.scd1_agr_terms t
                    where cast(t.n_agr as string) = cast(a.n_agr as string)
                      and cast(t.d_valid_from as date) <= cast('{m_end}' as date)
                      and (t.d_valid_to is null or cast(t.d_valid_to as date) > cast('{m_start}' as date))
                      and upper(trim(cast(t.cf_ter_type as string))) = 'P'
                      and coalesce(t.ods_deleted_flg, '0') <> '1'
              )
        """)

    if lake_m is None:
        lake_m = pd.DataFrame(columns=['agr_id', 'c_inn'])

    lake_inn_set_m = set(lake_m['c_inn'].apply(normalize_inn).dropna().drop_duplicates().tolist())
    lake_agr_set_m = set(lake_m['agr_id'].apply(normalize_agr).dropna().drop_duplicates().tolist())

    inn_intersect_m = excel_inn_set_m & lake_inn_set_m
    agr_intersect_m = excel_agr_set_m & lake_agr_set_m

    only_lake_inn_m = lake_inn_set_m - excel_inn_set_m
    only_excel_inn_m = excel_inn_set_m - lake_inn_set_m
    only_lake_agr_m = lake_agr_set_m - excel_agr_set_m
    only_excel_agr_m = excel_agr_set_m - lake_agr_set_m

    q1_only_lake_inn[report_month] = sorted(list(only_lake_inn_m))
    q1_only_excel_inn[report_month] = sorted(list(only_excel_inn_m))
    q1_only_lake_agr[report_month] = sorted(list(only_lake_agr_m))
    q1_only_excel_agr[report_month] = sorted(list(only_excel_agr_m))

    q1_kpi_rows.append({
        'report_month': report_month,
        'excel_inn_cnt': len(excel_inn_set_m),
        'lake_inn_cnt_no_rshb': len(lake_inn_set_m),
        'inn_intersect_cnt': len(inn_intersect_m),
        'only_lake_inn_cnt': len(only_lake_inn_m),
        'only_excel_inn_cnt': len(only_excel_inn_m),
        'excel_agr_cnt': len(excel_agr_set_m),
        'lake_agr_cnt_no_rshb': len(lake_agr_set_m),
        'agr_intersect_cnt': len(agr_intersect_m),
        'only_lake_agr_cnt': len(only_lake_agr_m),
        'only_excel_agr_cnt': len(only_excel_agr_m),
    })

q1_kpi_df = pd.DataFrame(q1_kpi_rows).sort_values('report_month').reset_index(drop=True)
display(q1_kpi_df)

In [ ]:
# Шаг 9: примеры расхождений по месяцам — ИНН/agr_id только в Lake и только в Excel.
for m in sorted(q1_only_lake_inn.keys()):
    print(f"\n===== {m} =====")

    print(f"ИНН только в Lake (без RSHB): {len(q1_only_lake_inn[m])}")
    display(pd.DataFrame({'inn_only_lake': q1_only_lake_inn[m]}).head(30))

    print(f"ИНН только в Excel: {len(q1_only_excel_inn[m])}")
    display(pd.DataFrame({'inn_only_excel': q1_only_excel_inn[m]}).head(30))

    print(f"agr_id только в Lake (без RSHB): {len(q1_only_lake_agr[m])}")
    display(pd.DataFrame({'agr_id_only_lake': q1_only_lake_agr[m]}).head(30))

    print(f"agr_id только в Excel: {len(q1_only_excel_agr[m])}")
    display(pd.DataFrame({'agr_id_only_excel': q1_only_excel_agr[m]}).head(30))

In [ ]:
# Шаг 10: удобный вывод списками для копирования (по месяцам).
for m in sorted(q1_only_lake_inn.keys()):
    print(f"\n===== {m} | COPY-LISTS =====")

    print('\nИНН только в Excel:')
    for x in q1_only_excel_inn[m]:
        print(x)

    print('\nagr_id только в Excel:')
    for x in q1_only_excel_agr[m]:
        print(x)

    print('\nagr_id только в Lake (без RSHB):')
    for x in q1_only_lake_agr[m]:
        print(x)

In [ ]:
# Шаг 11: диагностика — где отрезаются only_excel ИНН и agr_id по этапам фильтрации (Q1).
q1_cutoff_rows = []

for src in q1_sources:
    report_month = src['report_month']
    report_ts = pd.to_datetime(report_month)
    m_start = report_ts.strftime('%Y-%m-%d')
    m_end = (report_ts + pd.offsets.MonthEnd(1)).strftime('%Y-%m-%d')

    excel_raw_m = pd.read_excel(src['excel_path'], header=src['header'])
    excel_inn_col_m = pick_col(excel_raw_m.columns, inn_aliases)
    excel_agr_col_m = pick_col(excel_raw_m.columns, agr_aliases)

    excel_inn_set_m = set(excel_raw_m[excel_inn_col_m].apply(normalize_inn).dropna().drop_duplicates().tolist())
    excel_agr_set_m = set(excel_raw_m[excel_agr_col_m].apply(normalize_agr).dropna().drop_duplicates().tolist())

    with imp:
        s1 = imp.fetch(f"""
            select distinct cast(c.c_inn as string) as c_inn, cast(a.abs_agr_id as string) as agr_id
            from ods_alpha.scd1_agreements a
            join ods_alpha.scd1_companies c on c.n_cmp = a.n_cmp_client
            where upper(trim(cast(a.acq_class as string))) = 'SA'
              and cast(a.d_valid_from as date) <= cast('{m_end}' as date)
              and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('{m_start}' as date))
              and coalesce(a.ods_deleted_flg, '0') <> '1'
              and coalesce(c.ods_deleted_flg, '0') <> '1'
              and c.c_inn is not null
        """)

        s2 = imp.fetch(f"""
            select distinct cast(c.c_inn as string) as c_inn, cast(a.abs_agr_id as string) as agr_id
            from ods_alpha.scd1_agreements a
            join ods_alpha.scd1_companies c on c.n_cmp = a.n_cmp_client
            where upper(trim(cast(a.acq_class as string))) = 'SA'
              and cast(a.d_valid_from as date) <= cast('{m_end}' as date)
              and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('{m_start}' as date))
              and coalesce(a.ods_deleted_flg, '0') <> '1'
              and coalesce(c.ods_deleted_flg, '0') <> '1'
              and c.c_inn is not null
              and exists (
                    select 1
                    from ods_alpha.scd1_agr_terms t
                    where cast(t.n_agr as string) = cast(a.n_agr as string)
                      and cast(t.d_valid_from as date) <= cast('{m_end}' as date)
                      and (t.d_valid_to is null or cast(t.d_valid_to as date) > cast('{m_start}' as date))
                      and coalesce(t.ods_deleted_flg, '0') <> '1'
              )
        """)

        s3 = imp.fetch(f"""
            select distinct cast(c.c_inn as string) as c_inn, cast(a.abs_agr_id as string) as agr_id
            from ods_alpha.scd1_agreements a
            join ods_alpha.scd1_companies c on c.n_cmp = a.n_cmp_client
            where upper(trim(cast(a.acq_class as string))) = 'SA'
              and cast(a.d_valid_from as date) <= cast('{m_end}' as date)
              and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('{m_start}' as date))
              and coalesce(a.ods_deleted_flg, '0') <> '1'
              and coalesce(c.ods_deleted_flg, '0') <> '1'
              and c.c_inn is not null
              and exists (
                    select 1
                    from ods_alpha.scd1_agr_terms t
                    where cast(t.n_agr as string) = cast(a.n_agr as string)
                      and cast(t.d_valid_from as date) <= cast('{m_end}' as date)
                      and (t.d_valid_to is null or cast(t.d_valid_to as date) > cast('{m_start}' as date))
                      and upper(trim(cast(t.cf_ter_type as string))) = 'P'
                      and coalesce(t.ods_deleted_flg, '0') <> '1'
              )
        """)

    for stage_name, stage_df in [
        ('stage1_agreements_sa_active', s1),
        ('stage2_plus_terms_active', s2),
        ('stage3_plus_cf_ter_type_P', s3),
    ]:
        if stage_df is None:
            stage_df = pd.DataFrame(columns=['c_inn', 'agr_id'])

        stage_inn_set = set(stage_df['c_inn'].apply(normalize_inn).dropna().drop_duplicates().tolist())
        stage_agr_set = set(stage_df['agr_id'].apply(normalize_agr).dropna().drop_duplicates().tolist())

        only_excel_inn_stage = excel_inn_set_m - stage_inn_set
        only_excel_agr_stage = excel_agr_set_m - stage_agr_set

        q1_cutoff_rows.append({
            'report_month': report_month,
            'stage': stage_name,
            'only_excel_inn_cnt': len(only_excel_inn_stage),
            'only_excel_agr_cnt': len(only_excel_agr_stage),
        })

q1_cutoff_df = pd.DataFrame(q1_cutoff_rows).sort_values(['report_month', 'stage']).reset_index(drop=True)
display(q1_cutoff_df)